# Silver Layer — Account & Entity Data

**Purpose:** Standardize the IBM AML accounts dataset, enforcing lowercase naming conventions for Gold layer compatibility.

In [ ]:
# ============================================================
# FinPulse — Silver Layer: IBM AML Account & Entity Data
# Purpose: Standardise account/entity reference data
# Storage: Apache Iceberg v2 Hadoop catalog
# Author: Amanjot Kaur
# ============================================================
import os
import sys
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

load_dotenv(override=True)

def _find_project_root(marker='pyproject.toml') -> Path:
    curr = Path(os.getcwd()).resolve()
    for cand in [curr] + list(curr.parents):
        if (cand / marker).exists(): return cand
    return curr

PROJECT_ROOT = _find_project_root()

# ── Clear bad SPARK_HOME ──────────────────────────────────────
if 'SPARK_HOME' in os.environ and not Path(os.environ['SPARK_HOME']).is_dir():
    del os.environ['SPARK_HOME']

# ── Environment ───────────────────────────────────────────────
os.environ['JAVA_HOME']             = r'C:\Program Files\Microsoft\jdk-11.0.30.7-hotspot'
os.environ['HADOOP_HOME']           = str(PROJECT_ROOT / 'hadoop')
os.environ['PYSPARK_PYTHON']        = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

for p in [
    Path(os.environ['JAVA_HOME']) / 'bin',
    Path(os.environ['HADOOP_HOME']) / 'bin',
]:
    if p.is_dir() and str(p) not in os.environ['PATH']:
        os.environ['PATH'] = str(p) + os.pathsep + os.environ['PATH']

# ── Paths ─────────────────────────────────────────────────────
BRONZE_ACCOUNTS   = str(PROJECT_ROOT / 'data' / 'bronze' / 'accounts')
ICEBERG_WAREHOUSE = os.environ.get(
    'ICEBERG_WAREHOUSE',
    str(PROJECT_ROOT / 'data' / 'silver' / 'iceberg_warehouse')
)
ICEBERG_JAR = str(PROJECT_ROOT / 'jars' /
    'iceberg-spark-runtime-3.5_2.12-1.4.3.jar')

PIPELINE_VERSION    = 'v2.0'
INGESTION_TIMESTAMP = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print(f'Project root     : {PROJECT_ROOT}')
print(f'Bronze path      : {BRONZE_ACCOUNTS}')
print(f'Iceberg warehouse: {ICEBERG_WAREHOUSE}')
print(f'JAR exists       : {Path(ICEBERG_JAR).exists()}')

# ── Spark + Iceberg ───────────────────────────────────────────
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StringType

TEMP_DIR = Path.home() / 'FinPulse_Spark_Temp'
os.makedirs(TEMP_DIR, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('FinPulse-Silver-Accounts')
    .master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.executor.memory', '2g')
    .config('spark.sql.shuffle.partitions', '4')
    # ── Windows critical fixes ────────────────────────────────
    .config('spark.driver.extraJavaOptions',
            f'-Dorg.apache.hadoop.io.nativeio.NativeIO.Windows.should_use_native_io=false'
            f' -Djava.io.tmpdir="{str(TEMP_DIR).replace(chr(92), "/")}"')
    .config('spark.executor.extraJavaOptions',
            '-Dorg.apache.hadoop.io.nativeio.NativeIO.Windows.should_use_native_io=false')
    .config('spark.hadoop.fs.file.impl',
            'org.apache.hadoop.fs.RawLocalFileSystem')
    .config('spark.hadoop.fs.file.impl.disable.cache', 'true')
    # ── Iceberg ───────────────────────────────────────────────
    .config('spark.jars', ICEBERG_JAR)
    .config('spark.sql.extensions',
            'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.sql.catalog.local',
            'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.local.type', 'hadoop')
    .config('spark.sql.catalog.local.warehouse', ICEBERG_WAREHOUSE)
    .config('spark.sql.defaultCatalog', 'local')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print(f'\n✅ Spark + Iceberg ready | version: {spark.version}')

In [ ]:
# Find latest partition
partitions = [d for d in os.listdir(BRONZE_ACCOUNTS) if d.startswith("date=")]
latest_partition = sorted(partitions)[-1]
bronze_file = Path(BRONZE_ACCOUNTS) / latest_partition / "accounts_raw.csv"

print(f"Loading: {bronze_file}")
df_raw = spark.read.csv(str(bronze_file), header=True, inferSchema=True)
print(f"Raw accounts: {df_raw.count():,}")

In [ ]:
def apply_account_transformations(df):
    print("=" * 60)
    print("APPLYING ACCOUNTS SILVER TRANSFORMATIONS")
    print("=" * 60)

    # Step 1 — Standardise column names to lowercase snake_case
    for col in df.columns:
        new_col = col.lower().replace(" ", "_")
        df = df.withColumnRenamed(col, new_col)
    print(f"Step 1 — Columns standardised: {df.columns}")

    # Step 2 — Clean entity_name and entity_type
    # Trim whitespace from string columns
    df = df.withColumn("entity_name", F.trim(F.col("entity_name")))
    if "entity_type" in df.columns:
        df = df.withColumn("entity_type", F.trim(F.col("entity_type")))
    print("Step 2 — entity_name and entity_type trimmed")

    # Step 3 — Categorise entity type
    # IBM AML has: Corporation, Partnership, Sole Proprietorship
    # Encode for ML readability
    df = df.withColumn(
        "entity_type_encoded",
        F.when(F.col("entity_name").contains("Corporation"), 0)
         .when(F.col("entity_name").contains("Partnership"), 1)
         .when(F.col("entity_name").contains("Sole Proprietorship"), 2)
         .otherwise(-1)
    )
    corp_count = df.filter(F.col("entity_type_encoded") == 0).count()
    part_count = df.filter(F.col("entity_type_encoded") == 1).count()
    sole_count = df.filter(F.col("entity_type_encoded") == 2).count()
    print(f"Step 3 — Entity types: Corporations={corp_count:,} "
          f"Partnerships={part_count:,} Sole Proprietorships={sole_count:,}")

    # Step 4 — Clean bank_name
    df = df.withColumn("bank_name", F.trim(F.col("bank_name")))
    unique_banks = df.select("bank_name").distinct().count()
    print(f"Step 4 — Unique banks: {unique_banks:,}")

    # Step 5 — Deduplicate on account_number
    before = df.count()
    df = df.dropDuplicates(["account_number"])
    dupes = before - df.count()
    print(f"Step 5 — Deduplicated: {dupes:,} duplicates removed")

    # Step 6 — Pipeline metadata
    df = df.withColumn("silver_processed_at", F.current_timestamp())
    df = df.withColumn("pipeline_version", F.lit("v2.0"))
    print("Step 6 — Pipeline metadata added")

    print(f"\n✅ Transformations complete")
    print(f"Final records: {df.count():,}")
    print(f"Final columns: {len(df.columns)}")
    return df

silver_df = apply_account_transformations(df_raw)
silver_df.printSchema()

In [ ]:
# ── Validation ───────────────────────────────────────────────
print("=" * 60)
print("ACCOUNTS SILVER VALIDATION")
print("=" * 60)

checks_passed = 0

# Check 1 — No null account numbers
null_accounts = silver_df.filter(F.col("account_number").isNull()).count()
if null_accounts == 0:
    print("✅ Check 1 PASSED — No null account numbers")
    checks_passed += 1
else:
    print(f"❌ Check 1 FAILED — {null_accounts} null account numbers")

# Check 2 — No null bank names
null_banks = silver_df.filter(F.col("bank_name").isNull()).count()
if null_banks == 0:
    print("✅ Check 2 PASSED — No null bank names")
    checks_passed += 1
else:
    print(f"❌ Check 2 FAILED — {null_banks} null bank names")

# Check 3 — Entity type breakdown
print("\n── Entity Type Breakdown ──")
silver_df.groupBy("entity_type_encoded").agg(
    F.count("*").alias("count")
).orderBy("entity_type_encoded").show()

print(f"Validation complete — {checks_passed}/2 passed")

# ── Write to Iceberg ──────────────────────────────────────────
print("\n" + "=" * 60)
print("WRITING ACCOUNTS SILVER — Apache Iceberg")
print("=" * 60)

spark.sql("CREATE NAMESPACE IF NOT EXISTS local.finpulse")
print("✅ Namespace local.finpulse ready")

silver_df.writeTo("local.finpulse.accounts_silver") \
    .tableProperty("format-version", "2") \
    .tableProperty("write.parquet.compression-codec", "snappy") \
    .createOrReplace()

print("✅ Iceberg write complete: local.finpulse.accounts_silver")

# ── Verify via file system ────────────────────────────────────
from pathlib import Path as _Path
iceberg_table_path = _Path(ICEBERG_WAREHOUSE) / "finpulse" / "accounts_silver" / "data"

if iceberg_table_path.exists():
    parquet_files = list(iceberg_table_path.rglob("*.parquet"))
    total_size = sum(f.stat().st_size for f in parquet_files) / (1024 * 1024)
    print(f"\n✅ Iceberg table verified on disk:")
    print(f"   Parquet files : {len(parquet_files)}")
    print(f"   Total size    : {total_size:.2f} MB")
    print(f"   Location      : {iceberg_table_path}")
else:
    print("❌ Iceberg data folder not found")

# ── Summary ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("FINPULSE SILVER ACCOUNTS — COMPLETE SUMMARY")
print("=" * 60)
print(f"Dataset         : IBM AML Account Reference Data")
print(f"Records         : {silver_df.count():,}")
print(f"Columns         : {len(silver_df.columns)}")
print(f"Transformations :")
print(f"  1. Column names standardised to snake_case")
print(f"  2. entity_name and entity_type trimmed")
print(f"  3. entity_type_encoded (0=Corp, 1=Partnership, 2=Sole Prop)")
print(f"  4. bank_name cleaned")
print(f"  5. Deduplicated on account_number")
print(f"  6. Pipeline metadata added")
print(f"Iceberg table   : local.finpulse.accounts_silver")
print(f"Storage format  : Apache Iceberg v2, Snappy compression")
print(f"Purpose         : Entity context for Gold layer 3-way join")
print("=" * 60)
print("✅ SILVER ACCOUNTS COMPLETE")
print("=" * 60)

spark.stop()
print("\n✅ Spark stopped")